In [1]:
# =========================================================
# STATISTICAL ANALYSIS
# =========================================================

from pathlib import Path

import pandas as pd
import numpy as np

from scipy import stats

import statsmodels.api as sm
import statsmodels.formula.api as smf


# =========================================================
# PROJECT PATH
# =========================================================

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cleaned_survey.csv"
)


if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found at:\n{DATA_PATH}"
    )


df = pd.read_csv(DATA_PATH)


print("=" * 80)
print("OUTLOOK CUSTOMER INTELLIGENCE")
print("STATISTICAL ANALYSIS")
print("=" * 80)

print(
    f"\nDataset: {df.shape[0]:,} rows × "
    f"{df.shape[1]:,} columns"
)

OUTLOOK CUSTOMER INTELLIGENCE
STATISTICAL ANALYSIS

Dataset: 11,109 rows × 101 columns


In [2]:
# =========================================================
# STATISTICAL HELPER FUNCTIONS
# =========================================================

def cramers_v(table):
    """
    Calculate Cramér's V for a contingency table.
    """

    chi2 = stats.chi2_contingency(table)[0]

    n = table.to_numpy().sum()

    r, k = table.shape

    phi2 = chi2 / n

    phi2_corrected = max(
        0,
        phi2 - ((k - 1) * (r - 1)) / (n - 1)
    )

    r_corrected = (
        r
        - ((r - 1) ** 2) / (n - 1)
    )

    k_corrected = (
        k
        - ((k - 1) ** 2) / (n - 1)
    )

    denominator = min(
        k_corrected - 1,
        r_corrected - 1
    )

    if denominator <= 0:
        return np.nan

    return np.sqrt(
        phi2_corrected / denominator
    )


def interpret_cramers_v(v):
    """
    Rough interpretation of Cramér's V.
    These are guidelines rather than universal thresholds.
    """

    if pd.isna(v):
        return "Not available"

    if v < 0.10:
        return "Very weak"

    if v < 0.20:
        return "Weak"

    if v < 0.30:
        return "Moderate"

    return "Strong"


def interpret_spearman(rho):
    """
    Rough interpretation of absolute Spearman correlation.
    """

    strength = abs(rho)

    if strength < 0.10:
        label = "Very weak"

    elif strength < 0.30:
        label = "Weak"

    elif strength < 0.50:
        label = "Moderate"

    elif strength < 0.70:
        label = "Strong"

    else:
        label = "Very strong"

    direction = (
        "positive"
        if rho >= 0
        else "negative"
    )

    return f"{label} {direction}"

In [3]:
# =========================================================
# CHI-SQUARE:
# AWARENESS × DISCOVERY METHOD
# =========================================================

table = pd.crosstab(
    df["brand_discovery_method"],
    df["outlook_awareness_1"]
)

chi2, p_value, degrees_freedom, expected = (
    stats.chi2_contingency(table)
)

v = cramers_v(table)

print("CONTINGENCY TABLE")
print(table)

print("\nCHI-SQUARE TEST")
print(f"Chi-square : {chi2:.4f}")
print(f"Degrees of freedom : {degrees_freedom}")
print(f"P-value : {p_value:.6f}")
print(f"Cramér's V : {v:.4f}")
print(
    f"Effect interpretation : "
    f"{interpret_cramers_v(v)}"
)

CONTINGENCY TABLE
outlook_awareness_1          No   Yes
brand_discovery_method               
I myself read it           1121  1165
Others                     1121  1055
People who I know read it  1173  1070
Print Media                1092  1090
Social Media               1116  1106

CHI-SQUARE TEST
Chi-square : 5.9368
Degrees of freedom : 4
P-value : 0.203920
Cramér's V : 0.0132
Effect interpretation : Very weak


In [4]:
# =========================================================
# CHI-SQUARE TESTS FOR AWARENESS
# =========================================================

categorical_tests = [
    (
        "Discovery Method",
        "brand_discovery_method"
    ),
    (
        "Digital Ad Platform",
        "digital_ad_platform"
    ),
    (
        "Campaign Frequency",
        "digital_campaign_frequency"
    ),
    (
        "Digital Purchase Frequency",
        "digital_purchase_frequency"
    ),
    (
        "Digital Monthly Spend",
        "digital_monthly_spend"
    )
]


awareness_test_results = []


for label, column in categorical_tests:

    table = pd.crosstab(
        df[column],
        df["outlook_awareness_1"]
    )

    chi2, p_value, dof, expected = (
        stats.chi2_contingency(table)
    )

    v = cramers_v(table)

    awareness_test_results.append({
        "variable": label,
        "chi_square": chi2,
        "degrees_of_freedom": dof,
        "p_value": p_value,
        "cramers_v": v,
        "effect": interpret_cramers_v(v)
    })


awareness_test_results = pd.DataFrame(
    awareness_test_results
)

awareness_test_results.round(4)

,variable,chi_square,degrees_of_freedom,p_value,cramers_v,effect
0,Discovery Method,5.9368,4,0.2039,0.0132,Very weak
1,Digital Ad Platform,8.9099,11,0.6302,0.0000,Very weak
2,Campaign Frequency,1.0189,3,0.7967,0.0000,Very weak
3,Digital Purchase Frequency,8.3380,4,0.0800,0.0198,Very weak
4,Digital Monthly Spend,3.6108,4,0.4612,0.0000,Very weak


In [5]:
# =========================================================
# CHECK CHI-SQUARE EXPECTED COUNTS
# =========================================================

for label, column in categorical_tests:

    table = pd.crosstab(
        df[column],
        df["outlook_awareness_1"]
    )

    _, _, _, expected = stats.chi2_contingency(
        table
    )

    minimum_expected = expected.min()

    print(
        f"{label}: "
        f"minimum expected count = "
        f"{minimum_expected:.2f}"
    )

Discovery Method: minimum expected count = 1074.58
Digital Ad Platform: minimum expected count = 425.19
Campaign Frequency: minimum expected count = 1359.52
Digital Purchase Frequency: minimum expected count = 1087.92
Digital Monthly Spend: minimum expected count = 1074.58


In [6]:
# =========================================================
# SPEARMAN CORRELATIONS
# =========================================================

target = "recommendation"

predictors = [
    "service_rating",
    "content_rating",
    "price_rating",
    "delivery_rating",
    "previous_experience_rating",
    "customer_support_rating",
    "brand_solution_rating",
    "overall_magazine_rating",
    "overall_experience_rating"
]


spearman_results = []


for column in predictors:

    rho, p_value = stats.spearmanr(
        df[target],
        df[column]
    )

    spearman_results.append({
        "variable": column,
        "rho": rho,
        "p_value": p_value,
        "interpretation": interpret_spearman(rho)
    })


spearman_results = (
    pd.DataFrame(spearman_results)
    .sort_values(
        "p_value"
    )
)


spearman_results.round(4)

,variable,rho,p_value,interpretation
1,content_rating,0.0140,0.1389,Very weak positive
3,delivery_rating,0.0084,0.3753,Very weak positive
2,price_rating,-0.0061,0.5212,Very weak negative
7,overall_magazine_rating,0.0051,0.5943,Very weak positive
6,brand_solution_rating,0.0034,0.7240,Very weak positive
4,previous_experience_rating,-0.0026,0.7810,Very weak negative
0,service_rating,-0.0016,0.8649,Very weak negative
5,customer_support_rating,-0.0015,0.8706,Very weak negative
8,overall_experience_rating,-0.0012,0.8986,Very weak negative


In [7]:
# =========================================================
# REPURCHASE INTENT × EXPERIENCE
# =========================================================

repurchase_spearman_results = []


for column in predictors:

    rho, p_value = stats.spearmanr(
        df["repurchase_intent_score"],
        df[column]
    )

    repurchase_spearman_results.append({
        "variable": column,
        "rho": rho,
        "p_value": p_value,
        "interpretation": interpret_spearman(rho)
    })


repurchase_spearman_results = (
    pd.DataFrame(
        repurchase_spearman_results
    )
    .sort_values("p_value")
)


repurchase_spearman_results.round(4)

,variable,rho,p_value,interpretation
6,brand_solution_rating,-0.0212,0.0256,Very weak negative
0,service_rating,-0.0189,0.0462,Very weak negative
3,delivery_rating,0.0136,0.1508,Very weak positive
8,overall_experience_rating,0.0115,0.2260,Very weak positive
2,price_rating,-0.0103,0.2774,Very weak negative
1,content_rating,0.0067,0.4822,Very weak positive
5,customer_support_rating,-0.0023,0.8062,Very weak negative
7,overall_magazine_rating,0.0011,0.9092,Very weak positive
4,previous_experience_rating,-0.0001,0.9919,Very weak negative


In [8]:
# =========================================================
# COHEN'S KAPPA:
# AWARENESS QUESTION AGREEMENT
# =========================================================

awareness_1 = (
    df["outlook_awareness_1"]
    .str.strip()
    .str.lower()
)

awareness_2 = (
    df["outlook_awareness_2"]
    .str.strip()
    .str.lower()
)


kappa_table = pd.crosstab(
    awareness_1,
    awareness_2
)


observed_agreement = (
    np.trace(kappa_table.values)
    / kappa_table.values.sum()
)


row_totals = kappa_table.sum(axis=1).values
column_totals = kappa_table.sum(axis=0).values
n = kappa_table.values.sum()


expected_agreement = (
    row_totals @ column_totals
    / n**2
)


cohens_kappa = (
    observed_agreement
    - expected_agreement
) / (
    1 - expected_agreement
)


print("Awareness agreement table:")
print(kappa_table)

print(
    f"\nObserved agreement: "
    f"{observed_agreement:.4f}"
)

print(
    f"Expected agreement by chance: "
    f"{expected_agreement:.4f}"
)

print(
    f"Cohen's Kappa: "
    f"{cohens_kappa:.4f}"
)

Awareness agreement table:
outlook_awareness_2    no   yes
outlook_awareness_1            
no                   2838  2785
yes                  2785  2701

Observed agreement: 0.4986
Expected agreement by chance: 0.5001
Cohen's Kappa: -0.0029


In [9]:
# =========================================================
# HOLM MULTIPLE-TESTING CORRECTION
# =========================================================

from statsmodels.stats.multitest import multipletests


# ---------------------------------------------------------
# AWARENESS TESTS
# ---------------------------------------------------------

awareness_test_results["p_value_adjusted"] = (
    multipletests(
        awareness_test_results["p_value"],
        method="holm"
    )[1]
)

awareness_test_results["significant_after_holm"] = (
    awareness_test_results["p_value_adjusted"] < 0.05
)


print("AWARENESS TESTS")
print(
    awareness_test_results[
        [
            "variable",
            "p_value",
            "p_value_adjusted",
            "cramers_v",
            "effect",
            "significant_after_holm"
        ]
    ].round(4)
)


# ---------------------------------------------------------
# REPURCHASE SPEARMAN TESTS
# ---------------------------------------------------------

repurchase_spearman_results["p_value_adjusted"] = (
    multipletests(
        repurchase_spearman_results["p_value"],
        method="holm"
    )[1]
)

repurchase_spearman_results["significant_after_holm"] = (
    repurchase_spearman_results["p_value_adjusted"] < 0.05
)


print("\nREPURCHASE SPEARMAN TESTS")

print(
    repurchase_spearman_results[
        [
            "variable",
            "rho",
            "p_value",
            "p_value_adjusted",
            "interpretation",
            "significant_after_holm"
        ]
    ].round(4)
)

AWARENESS TESTS
                     variable  p_value  p_value_adjusted  cramers_v  \
0            Discovery Method   0.2039            0.8157     0.0132   
1         Digital Ad Platform   0.6302            1.0000     0.0000   
2          Campaign Frequency   0.7967            1.0000     0.0000   
3  Digital Purchase Frequency   0.0800            0.3998     0.0198   
4       Digital Monthly Spend   0.4612            1.0000     0.0000   

      effect  significant_after_holm  
0  Very weak                   False  
1  Very weak                   False  
2  Very weak                   False  
3  Very weak                   False  
4  Very weak                   False  

REPURCHASE SPEARMAN TESTS
                     variable     rho  p_value  p_value_adjusted  \
6       brand_solution_rating -0.0212   0.0256            0.2304   
0              service_rating -0.0189   0.0462            0.3699   
3             delivery_rating  0.0136   0.1508            1.0000   
8   overall_experience_r

In [10]:
# =========================================================
# KRUSKAL-WALLIS TESTS
# REPURCHASE INTENT × EXPERIENCE
# =========================================================

experience_columns = [
    "service_rating",
    "content_rating",
    "price_rating",
    "delivery_rating",
    "previous_experience_rating",
    "customer_support_rating"
]


kruskal_results = []


groups = [
    df.loc[
        df["repurchase_intent"] == category,
        "service_rating"
    ]
    for category in [
        "No",
        "Will think of it",
        "Yes"
    ]
]


for column in experience_columns:

    groups = [
        df.loc[
            df["repurchase_intent"] == category,
            column
        ].dropna()
        for category in [
            "No",
            "Will think of it",
            "Yes"
        ]
    ]

    statistic, p_value = stats.kruskal(
        *groups
    )

    kruskal_results.append({
        "variable": column,
        "h_statistic": statistic,
        "p_value": p_value
    })


kruskal_results = pd.DataFrame(
    kruskal_results
)


# Holm correction
kruskal_results["p_value_adjusted"] = (
    multipletests(
        kruskal_results["p_value"],
        method="holm"
    )[1]
)


kruskal_results["significant_after_holm"] = (
    kruskal_results["p_value_adjusted"] < 0.05
)


kruskal_results.round(4)

,variable,h_statistic,p_value,p_value_adjusted,significant_after_holm
0,service_rating,4.6182,0.0993,0.5961,False
1,content_rating,0.4953,0.7806,1.0000,False
2,price_rating,1.1832,0.5534,1.0000,False
3,delivery_rating,2.1425,0.3426,1.0000,False
4,previous_experience_rating,2.7237,0.2562,1.0000,False
5,customer_support_rating,0.3363,0.8452,1.0000,False


In [11]:
# =========================================================
# PAIRWISE MANN-WHITNEY TESTS
# =========================================================

from itertools import combinations


pairwise_results = []


repurchase_groups = [
    "No",
    "Will think of it",
    "Yes"
]


for column in experience_columns:

    overall_p = kruskal_results.loc[
        kruskal_results["variable"] == column,
        "p_value_adjusted"
    ].iloc[0]

    # Only perform pairwise testing when the
    # omnibus test is statistically significant.
    if overall_p < 0.05:

        for group1, group2 in combinations(
            repurchase_groups,
            2
        ):

            values1 = df.loc[
                df["repurchase_intent"] == group1,
                column
            ].dropna()

            values2 = df.loc[
                df["repurchase_intent"] == group2,
                column
            ].dropna()

            statistic, p_value = (
                stats.mannwhitneyu(
                    values1,
                    values2,
                    alternative="two-sided"
                )
            )

            pairwise_results.append({
                "variable": column,
                "group_1": group1,
                "group_2": group2,
                "u_statistic": statistic,
                "p_value": p_value
            })


pairwise_results = pd.DataFrame(
    pairwise_results
)


if not pairwise_results.empty:

    pairwise_results["p_value_adjusted"] = (
        multipletests(
            pairwise_results["p_value"],
            method="holm"
        )[1]
    )

    pairwise_results[
        "significant_after_holm"
    ] = (
        pairwise_results[
            "p_value_adjusted"
        ] < 0.05
    )

    print(
        pairwise_results.round(4)
    )

else:

    print(
        "No statistically significant "
        "Kruskal-Wallis tests were found. "
        "Therefore pairwise testing was not required."
    )

No statistically significant Kruskal-Wallis tests were found. Therefore pairwise testing was not required.


In [12]:
# =========================================================
# INTERPRET COHEN'S KAPPA
# =========================================================

def interpret_kappa(kappa):

    if kappa < 0:
        return "Poor agreement"

    elif kappa < 0.20:
        return "Slight agreement"

    elif kappa < 0.40:
        return "Fair agreement"

    elif kappa < 0.60:
        return "Moderate agreement"

    elif kappa < 0.80:
        return "Substantial agreement"

    else:
        return "Almost perfect agreement"


print(
    f"Cohen's Kappa: {cohens_kappa:.4f}"
)

print(
    "Interpretation:",
    interpret_kappa(cohens_kappa)
)

Cohen's Kappa: -0.0029
Interpretation: Poor agreement


In [13]:
# =========================================================
# STATISTICAL ANALYSIS SUMMARY
# =========================================================

print("=" * 80)
print("STATISTICAL ANALYSIS SUMMARY")
print("=" * 80)


print("\nAWARENESS — CHI-SQUARE")
print(
    awareness_test_results[
        [
            "variable",
            "chi_square",
            "p_value",
            "p_value_adjusted",
            "cramers_v",
            "significant_after_holm"
        ]
    ].round(4)
)


print("\nREPURCHASE — SPEARMAN")
print(
    repurchase_spearman_results[
        [
            "variable",
            "rho",
            "p_value",
            "p_value_adjusted",
            "significant_after_holm"
        ]
    ].round(4)
)


print("\nREPURCHASE — KRUSKAL-WALLIS")
print(
    kruskal_results[
        [
            "variable",
            "h_statistic",
            "p_value",
            "p_value_adjusted",
            "epsilon_squared",
            "significant_after_holm"
        ]
    ].round(4)
)


print("\nAWARENESS RELIABILITY")
print(
    f"Observed agreement : "
    f"{observed_agreement:.4f}"
)

print(
    f"Expected agreement : "
    f"{expected_agreement:.4f}"
)

print(
    f"Cohen's Kappa : "
    f"{cohens_kappa:.4f}"
)

print(
    f"Interpretation : "
    f"{interpret_kappa(cohens_kappa)}"
)

STATISTICAL ANALYSIS SUMMARY

AWARENESS — CHI-SQUARE
                     variable  chi_square  p_value  p_value_adjusted  \
0            Discovery Method      5.9368   0.2039            0.8157   
1         Digital Ad Platform      8.9099   0.6302            1.0000   
2          Campaign Frequency      1.0189   0.7967            1.0000   
3  Digital Purchase Frequency      8.3380   0.0800            0.3998   
4       Digital Monthly Spend      3.6108   0.4612            1.0000   

   cramers_v  significant_after_holm  
0     0.0132                   False  
1     0.0000                   False  
2     0.0000                   False  
3     0.0198                   False  
4     0.0000                   False  

REPURCHASE — SPEARMAN
                     variable     rho  p_value  p_value_adjusted  \
6       brand_solution_rating -0.0212   0.0256            0.2304   
0              service_rating -0.0189   0.0462            0.3699   
3             delivery_rating  0.0136   0.1508       

KeyError: "['epsilon_squared'] not in index"